# LA temp-market quoting arb: crossing table

**Strategy:** once the running-max temperature for the day has already passed a threshold, whichever side that guarantees is *locked in*:
- a "> strike" tail market (e.g. `>85`) is guaranteed **YES** once the running max exceeds its floor.
- a "< strike" tail market (e.g. `<80`) is guaranteed **NO** once the running max exceeds that same number (the day's final high can't end up below a value it's already passed).
- a "between" bucket market (e.g. `82-83`) is guaranteed **NO** once the running max exceeds its cap (the day's final high can no longer land in that bucket).

Resting a bid at 99c on the guaranteed side of any of these is close to risk-free. (The mirror cases -- `<80` guaranteed YES, `>85`/a bucket guaranteed YES from having already been passed on the way down -- aren't modeled, since those need knowing there's no time left in the day for the max to still move, not just where the running max currently is.)

**What this notebook does:** for every cached KXHIGHLAX day and every market that day, find when the real station temperature first crossed that market's own guarantee threshold (with two robustness guards, see below), then build one row per crossing: date, time of crossing, the market's range, the day's actual final max temp (so you can eyeball rounding-boundary cases yourself), how many lots were already resting at the 99c guaranteed-side bid right at that moment, how many lots total traded on that market (at any price) afterward, and when the 1st/10th/100th/1000th/10000th lot resting at the 99c bid actually got filled.

**Notes:**
- Station data is IEM's 1-minute ASOS product -- see `station_temp_data.fetch_station_temps`'s docstring. It has real per-day gaps (some days return zero rows); those days simply can't trigger a crossing.
- A crossing must clear the threshold by `ROUNDING_BUFFER_F` (guards against the confirmed ~1F disagreement between this trace and the official CLI settlement) AND sustain it for `MIN_CONSECUTIVE_OBS` readings (guards against a single anomalous/noisy tick) before it counts -- see `station_temp_data.first_persistent_crossing_time`'s docstring.
- `lots_at_99_bid_on_cross` comes from Predexon's tick-level orderbook (has known gaps). `lots_taken_after_cross` and the `t_qp{n}` fill-time columns come from Kalshi's own trade history, which only retains roughly the last ~2.5 months -- `RUN_DAYS` is already restricted to that window.

In [ ]:
import sys, os, datetime
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'kalshi_infra'))

import pandas as pd
import api_functions
import predexon_functions as pf
import station_temp_data as std

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

SERIES = 'KXHIGHLAX'
FILL_BID = 0.99  # our resting bid price

# Two independent guards against being "spoofed" by a bad temperature
# reading -- they defend against different failure modes:
# - ROUNDING_BUFFER_F: the station trace (IEM's 1-minute ASOS product) can
#   read up to ~1F off the eventual official CLI settlement value
#   (confirmed directly) -- clearing the threshold by a full degree means
#   that disagreement alone can't flip a real crossing into a false one.
# - MIN_CONSECUTIVE_OBS: a single anomalous/noisy reading shouldn't be
#   trusted alone -- require the crossing to be confirmed by at least this
#   many consecutive readings.
ROUNDING_BUFFER_F = 1.0
MIN_CONSECUTIVE_OBS = 2

# Queue depths (in lots) simulated for the t_qp{n} columns -- see that cell
# for what these mean.
QUEUE_DEPTHS = [1000, 10000, 50000, 100000, 200000, 300000]

src = std.get_kalshi_settlement_source(SERIES)
print('Settlement source (from Kalshi itself, not hardcoded):', src)

In [ ]:
# Every day we have official Kalshi 1-min bid/ask history for (needed to check fills).
def _event_date(event_ticker):
    return datetime.datetime.strptime(event_ticker.split('-', 1)[1], '%y%b%d').date()

all_event_tickers = [et for et in pf.all_cached_kalshi_1min_event_tickers() if et.startswith(f'{SERIES}-')]
days = sorted(_event_date(et) for et in all_event_tickers)
print(f'{len(days)} cached {SERIES} days, {days[0]} .. {days[-1]}')

# Kalshi's trades endpoint (get_market_trades) only retains ~2.5 months --
# confirmed directly by date-bisecting: 80 days back from today had 0
# trades, 75 days back had real data. Restrict to days inside that window
# so lots_taken_after_cross is a real number for every row instead of NaN
# for anything older, and so we're not wastefully fetching station/market
# data for days that can never get a trades answer anyway.
TRADES_LOOKBACK_DAYS = 75
trades_cutoff = datetime.date.today() - datetime.timedelta(days=TRADES_LOOKBACK_DAYS)
RUN_DAYS = [d for d in days if d >= trades_cutoff]
print(f'{len(RUN_DAYS)}/{len(days)} days are within the trades retention window (>= {trades_cutoff})')

In [ ]:
# Fetch (cached after first run) the official daily CLI settlement report and the
# intraday station trace for every day. Network calls only happen once per day ever.
station_data = {}
n_warn = 0
for d in RUN_DAYS:
    try:
        cli = std.fetch_cli_report(d, wfo=src['wfo'], station=src['station'])
        trace = std.fetch_station_temps(d, station=src['station'])
        station_data[d] = (cli, trace)
    except Exception as e:
        n_warn += 1
        print(f'  [warn] {d}: {e}')
print(f'{len(station_data)}/{len(RUN_DAYS)} days with station data ({n_warn} warnings)')

n_no_trace = sum(1 for _, trace in station_data.values() if trace.empty)
print(f'{n_no_trace}/{len(station_data)} days have NO 1-minute station data at all (real per-day IEM gap) -- '
      f'those days simply can\'t trigger a crossing, not treated as zero-opportunity')

In [ ]:
# One row per (day, market): a tail market is guaranteed once the running max
# crosses its own threshold in the direction that rules it out --
# "> strike" is guaranteed YES once the running max exceeds its floor;
# "< strike" is guaranteed NO once the running max exceeds that same
# number (the day's final high can't end up below a value it's already
# passed). A "between" bucket is guaranteed NO once the running max exceeds
# its cap (the final high can't land in a bucket already passed). All three
# are threshold-crossing signals built the same way via
# first_persistent_crossing_time -- only which threshold and which
# settlement direction differs. (The mirror case -- "< strike" guaranteed
# YES, "> strike"/bucket guaranteed YES from the far side -- needs knowing
# there's no time left in the day for the max to still rise, not just where
# the running max currently is, so it's not a simple crossing and isn't
# modeled here.)
#
# A crossing must clear the threshold by ROUNDING_BUFFER_F AND sustain it for
# MIN_CONSECUTIVE_OBS readings before counting (see config cell above) --
# either guard alone would let momentary station noise count as real.
opportunities = []
for d in RUN_DAYS:
    if d not in station_data:
        continue
    cli, trace = station_data[d]
    event_ticker = api_functions.event_ticker_from_date(d, series_ticker=SERIES)
    markets = pf.get_event_markets(event_ticker)
    for m in markets:
        floor, cap = pf._parse_strike_from_ticker(m['ticker'])
        strike_type = m.get('strike_type')
        if strike_type == 'greater':
            guaranteed_side, crossing_strike = 'yes', floor
        elif strike_type == 'less':
            guaranteed_side, crossing_strike = 'no', floor
        elif strike_type == 'between' and cap is not None:
            guaranteed_side, crossing_strike = 'no', cap
        else:
            continue
        t_cross = std.first_persistent_crossing_time(
            trace, crossing_strike, kind='max', min_consecutive=MIN_CONSECUTIVE_OBS, buffer=ROUNDING_BUFFER_F
        )
        opportunities.append({
            'date': d, 'event_ticker': event_ticker, 'ticker': m['ticker'],
            'strike_type': strike_type, 'guaranteed_side': guaranteed_side,
            'crossing_strike': crossing_strike, 'floor': floor, 'cap': cap,
            'settled_result': m.get('result'),
            't_cross_local': t_cross,
            'cli_max_f': cli['max_f'], 'cli_max_time_local': cli['max_time_local'],
        })

opp_df = pd.DataFrame(opportunities)
print(f'{len(opp_df)} candidate (day, market) rows')

triggered = opp_df[opp_df['t_cross_local'].notna()].copy()
print(f'{len(triggered)} triggered the arb ({ROUNDING_BUFFER_F}F buffer, confirmed by {MIN_CONSECUTIVE_OBS} consecutive readings)')
print(triggered.groupby(['strike_type', 'guaranteed_side']).size())

In [ ]:
def resting_qty_at_99(long_df, ticker, t_cross_local, guaranteed_side, price=FILL_BID):
    """Lots resting on the guaranteed side's bid at exactly `price`, read
    from the first cached Predexon quote at-or-after the crossing (only
    cached source with a depth field -- Predexon has known real gaps, see
    load_kalshi_backup_long_df's docstring). `long_df` is the whole event's
    Predexon quotes (load_predexon_long_df), passed in so it's only read
    from disk once per event rather than once per ticker. Predexon's
    bid/ask/bid_depth/ask_depth are always YES-side quotes (see
    load_predexon_long_df's docstring): a YES-side resting bid reads off
    bid/bid_depth directly; a NO-side resting bid is the same physical
    order as a YES ask quoted at (1 - price), so it reads off ask/ask_depth
    instead.

    None = no cached Predexon quote at all at/after the crossing (nothing
    to answer with). 0.0 = quoted, but nothing resting at `price` yet.
    """
    sub = long_df[long_df['ticker'] == ticker].sort_values('time')
    if sub.empty:
        return None
    t_cross_utc = pd.Timestamp(t_cross_local).tz_convert('UTC')
    after = sub[sub['time'] >= t_cross_utc]
    if after.empty:
        return None
    first = after.iloc[0]
    if guaranteed_side == 'yes':
        return float(first['bid_depth']) if abs(first['bid'] - price) < 1e-9 else 0.0
    else:
        return float(first['ask_depth']) if abs(first['ask'] - (1 - price)) < 1e-9 else 0.0


def lots_taken_after_cross(trades_df, ticker, t_cross_local):
    """Total contracts actually traded on this market, at ANY price, at or
    after the crossing -- from Kalshi's own trade history
    (api_functions.get_market_trades). `trades_df` is the whole event's
    trades (load_kalshi_trades_long_df), passed in so it's only read from
    disk once per event rather than once per ticker.

    None = trades_df is empty for the whole event, which almost always
    means this event predates Kalshi's ~2.5-month trade-history retention
    window (see load_kalshi_trades_long_df's docstring) -- not a confirmed
    zero. 0.0 = the event has trade data, but this ticker had no trades
    after the crossing.
    """
    if trades_df.empty:
        return None
    sub = trades_df[trades_df['ticker'] == ticker]
    t_cross_utc = pd.Timestamp(t_cross_local).tz_convert('UTC')
    after = sub[sub['time'] >= t_cross_utc]
    return float(after['count'].sum())


def lots_taken_at_99_between(trades_df, ticker, guaranteed_side, t1, t2, price=FILL_BID):
    """Total contracts traded at EXACTLY `price` on the guaranteed side
    between `t1` (a t_qp resting-depth time) and `t2` (the crossing) --
    only when t1 is AT OR BEFORE the crossing (the resting queue had
    already reached that depth BY the time of crossing, so "how much
    traded on the way there" is a real window). If t1 is AFTER the
    crossing (the depth wasn't reached until later), this is 0, not the
    trade volume in that forward window -- that's a different question
    (already covered by lots_taken_after_cross), not "how much was taken
    from the bid by crossing time." None if either time is missing or
    there's no trade data at all for this event.
    """
    if trades_df.empty or t1 is None or pd.isna(t1) or t2 is None or pd.isna(t2):
        return None
    if t1 > t2:
        return 0.0
    sub = trades_df[trades_df['ticker'] == ticker]
    price_col = 'yes_price' if guaranteed_side == 'yes' else 'no_price'
    at_price = sub[(sub[price_col] - price).abs() < 1e-9]
    window = at_price[(at_price['time'] >= t1) & (at_price['time'] <= t2)]
    return float(window['count'].sum())


def _resting_depth_slice(long_df, ticker, guaranteed_side, price=FILL_BID):
    """This ticker's Predexon quotes filtered to the guaranteed side's
    price==`price` rows, with the relevant depth column name -- shared
    setup for both times_resting_depth_reached and
    time_resting_depth_reached_single."""
    sub = long_df[long_df['ticker'] == ticker].sort_values('time')
    if guaranteed_side == 'yes':
        return sub[(sub['bid'] - price).abs() < 1e-9], 'bid_depth'
    else:
        return sub[(sub['ask'] - (1 - price)).abs() < 1e-9], 'ask_depth'


def times_resting_depth_reached(long_df, ticker, guaranteed_side, price=FILL_BID, depths=QUEUE_DEPTHS):
    """
    For each depth N in `depths`: the first time the RESTING size at
    exactly `price` on the guaranteed side reached at least N lots --
    independent of the crossing, searched across this ticker's whole
    Predexon quote history (not trades -- this is about how deep the
    resting queue itself got, not how much actually traded away; those two
    can differ hugely, e.g. a deep resting bid that few sellers ever hit).

    Returns a dict {n: timestamp_or_None for n in depths}. None means that
    depth of resting size was never reached, or there's no Predexon
    coverage for this ticker at all (known real gaps -- see
    load_predexon_long_df's docstring).
    """
    result = {n: None for n in depths}
    at_price, depth_col = _resting_depth_slice(long_df, ticker, guaranteed_side, price)
    if at_price.empty:
        return result
    for n in depths:
        hit = at_price[at_price[depth_col] >= n]
        if not hit.empty:
            result[n] = hit.iloc[0]['time']
    return result


def time_resting_depth_reached_single(long_df, ticker, guaranteed_side, n, price=FILL_BID):
    """Same as times_resting_depth_reached, but for a single depth `n`
    (used for the lots_taken_after_cross-sized depth, which is a different,
    row-specific number rather than one of the fixed QUEUE_DEPTHS). None if
    `n` is None/NaN or that depth was never reached."""
    if n is None or pd.isna(n):
        return None
    at_price, depth_col = _resting_depth_slice(long_df, ticker, guaranteed_side, price)
    if at_price.empty:
        return None
    hit = at_price[at_price[depth_col] >= n]
    return hit.iloc[0]['time'] if not hit.empty else None


def depth_label(n):
    """'1k' for 1000, '300k' for 300000 -- falls back to the plain integer
    string if n isn't a clean multiple of 1000."""
    n = int(n)
    return f'{n // 1000}k' if n % 1000 == 0 else str(n)


def market_range_str(strike_type, floor, cap):
    """The market's actual displayed range, e.g. '82-83' for a between
    bucket (floor=82, cap=83), '>80' for a greater-tail (floor=80,
    cap=None), or '<80' for a less-tail (floor=80, cap=None) -- not just
    the single crossing_strike threshold, which only captures the number
    relevant to the crossing and loses which market that was."""
    if strike_type == 'greater':
        return f'>{floor:.0f}'
    if strike_type == 'less':
        return f'<{floor:.0f}'
    return f'{floor:.0f}-{cap:.0f}'


DISPLAY_TZ = 'America/Chicago'  # CST/CDT -- handles the DST switch automatically


def hhmmss_cst(t):
    """HH:MM:SS in Central time (DISPLAY_TZ), or None if t is None/NaT."""
    if t is None or pd.isna(t):
        return None
    return pd.Timestamp(t).tz_convert(DISPLAY_TZ).strftime('%H:%M:%S')


def format_qp_cell(t, reference_utc, taken_between):
    """HH:MM:SS in Central time, plus:
    - a '(+N)'/'(-N)' suffix for how many CALENDAR days (in Central time)
      it falls from `reference_utc` (the crossing) -- negative means N
      days BEFORE the crossing, positive means N days AFTER, no suffix
      when same day, so a different-day value never silently looks
      same-day.
    - a '[N]' suffix showing lots actually traded at 99c between this time
      and the crossing (lots_taken_at_99_between) -- 0 if this time is
      AFTER the crossing (see that function's docstring).
    None if t is None/NaT.
    """
    if t is None or pd.isna(t):
        return None
    t_local = pd.Timestamp(t).tz_convert(DISPLAY_TZ)
    s = t_local.strftime('%H:%M:%S')
    ref_local = reference_utc.tz_convert(DISPLAY_TZ)
    day_offset = (t_local.normalize() - ref_local.normalize()).days
    if day_offset != 0:
        s += f' ({day_offset:+d})'
    if taken_between is not None and not pd.isna(taken_between):
        s += f' [{taken_between:,.2f}]'
    return s


# Cache each event's Predexon quotes / trade history once, keyed by
# event_ticker, instead of re-reading the same event's parquet from disk
# for every triggered market that day -- most days have several triggered
# between-bucket rows sharing one event.
_predexon_by_event = {}
_trades_by_event = {}

rows = []
for _, opp in triggered.iterrows():
    event_ticker = opp['event_ticker']
    if event_ticker not in _predexon_by_event:
        _predexon_by_event[event_ticker] = pf.load_predexon_long_df(event_ticker)
    if event_ticker not in _trades_by_event:
        _trades_by_event[event_ticker] = pf.load_kalshi_trades_long_df(event_ticker)
    long_df = _predexon_by_event[event_ticker]
    trades_df = _trades_by_event[event_ticker]
    ticker, side = opp['ticker'], opp['guaranteed_side']

    resting_qty = resting_qty_at_99(long_df, ticker, opp['t_cross_local'], side)
    taken_qty = lots_taken_after_cross(trades_df, ticker, opp['t_cross_local'])
    depth_times = times_resting_depth_reached(long_df, ticker, side)
    t_taken_qty = time_resting_depth_reached_single(long_df, ticker, side, taken_qty)
    t_cross_utc_ts = pd.Timestamp(opp['t_cross_local']).tz_convert('UTC')

    # 'date' + 'range' is the natural unique key now that neither ticker
    # nor guaranteed_side is a displayed column. Quote-time columns sit
    # right after t_cross_cst; size/volume columns follow.
    row = {
        'date': opp['date'],
        'range': market_range_str(opp['strike_type'], opp['floor'], opp['cap']),
        't_cross_cst': hhmmss_cst(opp['t_cross_local']),
    }
    for n in QUEUE_DEPTHS:
        t_n = depth_times[n]
        taken_between = lots_taken_at_99_between(trades_df, ticker, side, t_n, t_cross_utc_ts)
        row[f't_qp{depth_label(n)}'] = format_qp_cell(t_n, t_cross_utc_ts, taken_between)
    row['final_max_temp'] = opp['cli_max_f']
    row['lots_at_99_bid_on_cross'] = resting_qty
    row['lots_taken_after_cross'] = taken_qty
    taken_between_qty = lots_taken_at_99_between(trades_df, ticker, side, t_taken_qty, t_cross_utc_ts)
    row['t_qp_lots_taken_after_cross'] = format_qp_cell(t_taken_qty, t_cross_utc_ts, taken_between_qty)
    rows.append(row)

crossings_table = pd.DataFrame(rows)
assert not crossings_table.duplicated(subset=['date', 'range']).any(), \
    'duplicate (date, range) rows -- would double-count a crossing'
crossings_table = crossings_table.sort_values(['date', 't_cross_cst']).reset_index(drop=True)
crossings_table

## Caveats

- **All times are Central (`DISPLAY_TZ = 'America/Chicago'`, DST-aware).**
- **`lots_at_99_bid_on_cross` and the `t_qp*` columns' timestamps are quote-snapshot reads from Predexon's tick-level orderbook**, which has known real gaps (see `load_predexon_long_df`'s docstring).
- **Each `t_qp*` cell is `HH:MM:SS (+/-day offset) [lots taken between that time and the crossing]`.** The time is "when did the RESTING size at 99c first reach that many lots" -- not tied to the crossing (a `(-1)`/`(+1)` suffix shows the calendar-day offset from `t_cross_cst` when it isn't the same day; markets open ~1-2 days ahead). The bracketed number is real trade history: total lots actually traded at exactly 99c between that time and the crossing, from `load_kalshi_trades_long_df` -- **it's `0` whenever the depth was reached AFTER the crossing** (that's a forward-looking window, a different question already covered by `lots_taken_after_cross`); only computed as a real sum when the depth was reached at or before the crossing.
- **`t_qp1k`/`t_qp10k`/`t_qp50k`/`t_qp100k`/`t_qp200k`/`t_qp300k` use the fixed `QUEUE_DEPTHS`; `t_qp_lots_taken_after_cross` uses that row's own `lots_taken_after_cross` value as the depth instead.** Blank means that depth was never reached, or there's no Predexon coverage for this ticker at all.
- **`lots_taken_after_cross` is real trade history and IS crossing-relative** (total volume at any price, from the crossing to EOD), but only back to roughly Kalshi's ~2.5-month retention window -- `RUN_DAYS` is already restricted to that window, so a `0.0` there means genuinely no trades, not missing data.
- **LA only.** KXLOWTLAX has no historical bid/ask yet -- it'll accumulate via the ws collector going forward.
- **Fees not modeled.**